# CryptoHack: Elliptic Curves

Solutions to the **Elliptic Curves** course on [CryptoHack](https://cryptohack.org/courses/elliptic/course_details/).

The notebook starts by building elliptic curve arithmetic from scratch, then uses it to break
elliptic-curve cryptosystems whose parameters, implementations or signatures are flawed.

| Section | Attacks |
| --- | --- |
| Starter | Point arithmetic, ECDH |
| Parameter Choice | Pohlig-Hellman, Smart's attack (anomalous curves), small private keys, singular curves, MOV attack |
| Parameter Choice 2 | Elliptic curves over the reals and LLL |
| Signatures | ECDSA without hashing, CurveBall (CVE-2020-0601), tiny and biased nonces (hidden number problem) |
| Edwards Curves | Degenerate point decompression |
| Side Channels | Montgomery's ladder, power analysis of double-and-add |

The attacks themselves are implemented explicitly. For the heavy number theory that would be impractical to write
from scratch (counting points with the SEA algorithm, arithmetic in extension fields, pairings, LLL), we rely on
[PARI/GP](https://pari.math.u-bordeaux.fr/) through its Python bindings `cypari2`.

In [1]:
import hashlib
import json
import math
from collections import namedtuple

import cypari2
from Crypto.Cipher import AES
from Crypto.Util.number import bytes_to_long, long_to_bytes
from Crypto.Util.Padding import unpad
from sympy.ntheory import sqrt_mod

pari = cypari2.Pari()
pari.allocatemem(10**8, 2 * 10**9, silent=True)  # point counting on large curves needs a bigger stack


def decrypt_flag(shared_secret: int, iv: str, ciphertext: str) -> str:
    """Key derivation used by most challenges: AES key = SHA1(str(secret))[:16], AES-CBC."""
    key = hashlib.sha1(str(shared_secret).encode("ascii")).digest()[:16]
    plaintext = AES.new(key, AES.MODE_CBC, bytes.fromhex(iv)).decrypt(bytes.fromhex(ciphertext))
    try:
        plaintext = unpad(plaintext, AES.block_size)
    except ValueError:  # the servers do not pad a flag that already fills whole blocks
        pass
    return plaintext.decode("ascii")

# Starter

The starter challenges use the curve in short Weierstrass form

$$E: Y^2 = X^3 + 497X + 1768 \pmod{9739}$$

A curve is represented by its parameters `(a, b, p)`, a point by a tuple `(x, y)`, and the point at infinity `O`
(the identity element of the group) by `None`.

In [2]:
Curve = namedtuple("Curve", "a b p")
O = None  # the point at infinity

E = Curve(a=497, b=1768, p=9739)


def on_curve(E, P):
    return P is O or (P[1] ** 2 - (P[0] ** 3 + E.a * P[0] + E.b)) % E.p == 0

## Point Negation

The inverse of `P = (x, y)` is its reflection across the x-axis, `-P = (x, -y)`, so that `P + (-P) = O`.

In [3]:
def point_neg(E, P):
    return O if P is O else (P[0], -P[1] % E.p)


P = (8045, 6936)
Q = point_neg(E, P)
assert on_curve(E, P) and on_curve(E, Q)
print(f"crypto{{{Q[0]},{Q[1]}}}")

crypto{8045,2803}


## Point Addition

Adding `P = (x1, y1)` and `Q = (x2, y2)`:

- if `P = O` return `Q`, if `Q = O` return `P`;
- if `Q = -P`, then `P + Q = O`;
- otherwise compute the slope `λ`: `(y2 - y1)/(x2 - x1)` if `P ≠ Q`, or `(3x1² + a)/(2y1)` if `P = Q` (tangent);
- `x3 = λ² - x1 - x2` and `y3 = λ(x1 - x3) - y1`.

The flag is `S = P + P + Q + R`.

In [4]:
def point_add(E, P, Q):
    if P is O:
        return Q
    if Q is O:
        return P

    (x1, y1), (x2, y2) = P, Q
    if x1 == x2 and (y1 + y2) % E.p == 0:
        return O

    if P == Q:
        lam = (3 * x1 * x1 + E.a) * pow(2 * y1, -1, E.p) % E.p
    else:
        lam = (y2 - y1) * pow(x2 - x1, -1, E.p) % E.p

    x3 = (lam * lam - x1 - x2) % E.p
    y3 = (lam * (x1 - x3) - y1) % E.p
    return (x3, y3)


P = (493, 5564)
Q = (1539, 4742)
R = (4403, 5202)

S = point_add(E, point_add(E, P, P), point_add(E, Q, R))
assert on_curve(E, S)
print(S)

(4215, 2162)


## Scalar Multiplication

`n·P` is computed with the double-and-add algorithm, which needs `O(log n)` point operations.

In [5]:
def scalar_mult(E, P, n):
    if n < 0:
        return scalar_mult(E, point_neg(E, P), -n)
    result = O
    while n > 0:
        if n & 1:
            result = point_add(E, result, P)
        P = point_add(E, P, P)
        n >>= 1
    return result


Q = scalar_mult(E, (2339, 2213), 7863)
assert on_curve(E, Q)
print(Q)

(9467, 2742)


## Curves and Logs

Elliptic-curve Diffie-Hellman: the shared secret is `S = n_B · Q_A`.
The flag is the SHA-1 hash of the x-coordinate of `S`.

In [6]:
Q_A = (815, 3190)
n_B = 1829

S = scalar_mult(E, Q_A, n_B)
print(hashlib.sha1(str(S[0]).encode()).hexdigest())

80e5212754a824d3a4aed185ace4f9cac0f908bf


## Efficient Exchange

Alice only sends the x-coordinate of her public point. We recover `y` as a square root of `x³ + ax + b` modulo `p`.
Either root works, since `n·(x, y)` and `n·(x, -y)` share the same x-coordinate.
The shared secret's x-coordinate is then used to derive the AES key.

In [7]:
def lift_x(E, x):
    """Return a point of E with x-coordinate x."""
    y = sqrt_mod((x ** 3 + E.a * x + E.b) % E.p, E.p)
    if y is None:
        raise ValueError("x is not the x-coordinate of a point on the curve")
    return (x, y)


Q_A = lift_x(E, 4726)
n_B = 6534
data = {"iv": "cd9da9f1c60925922377ea952afc212c", "encrypted_flag": "febcbe3a3414a730b125931dccf912d2239f3e969c4334d95ed0ec86f6449ad8"}

shared_secret = scalar_mult(E, Q_A, n_B)[0]
print(decrypt_flag(shared_secret, data["iv"], data["encrypted_flag"]))

crypto{3ff1c1ent_k3y_3xch4ng3}


# Parameter Choice

## Smooth Criminal

The security of ECDH relies on the hardness of the elliptic curve discrete logarithm problem (ECDLP):
recovering `n` from `G` and `n·G`. Its difficulty depends on the **largest prime factor** of the order of `G`.

The **Pohlig-Hellman** algorithm splits the problem: for each prime power `q^e` dividing the order `N` of `G`,
it solves the logarithm in the subgroup of order `q^e` (generated by `(N / q^e)·G`), one base-`q` digit at a time,
with baby-step giant-step (BSGS). The results are recombined with the Chinese Remainder Theorem.

Here the curve order is *smooth*: its largest prime factor has only 38 bits, so each sub-problem is easy.
PARI counts the points of the curve with the SEA algorithm.

In [8]:
def curve_order(E):
    return int(pari(f"ellcard(ellinit([{E.a}, {E.b}], {E.p}))"))


def factor(n):
    return [(int(q), int(e)) for q, e in pari.factor(n).mattranspose()]


def point_order(E, P, n):
    """Order of P, given a multiple n of it (e.g. the curve order)."""
    for q, e in factor(n):
        for _ in range(e):
            if scalar_mult(E, P, n // q) is O:
                n //= q
    return n


def bsgs(E, G, Q, n):
    """Baby-step giant-step: find k < n such that Q = k*G, in O(sqrt(n)) operations."""
    m = math.isqrt(n) + 1
    baby_steps = {}
    P = O
    for j in range(m):
        baby_steps[P] = j
        P = point_add(E, P, G)
    giant_step = point_neg(E, scalar_mult(E, G, m))
    R = Q
    for i in range(m):
        if R in baby_steps:
            return (i * m + baby_steps[R]) % n
        R = point_add(E, R, giant_step)
    raise ValueError("logarithm not found")


def pohlig_hellman(E, G, Q, n, max_factor=None):
    """
    Find k such that Q = k*G, where n is the order of G.
    If max_factor is set, prime factors above it are skipped and k is only recovered modulo
    the product of the remaining prime powers. Return (k, modulus).
    """
    residues, moduli = [], []
    for q, e in factor(n):
        if max_factor and q > max_factor:
            continue
        G_i = scalar_mult(E, G, n // q ** e)  # order q^e
        Q_i = scalar_mult(E, Q, n // q ** e)
        gamma = scalar_mult(E, G_i, q ** (e - 1))  # order q
        k_i = 0
        for j in range(e):  # recover k_i one base-q digit at a time
            remainder = point_add(E, Q_i, point_neg(E, scalar_mult(E, G_i, k_i)))
            digit = bsgs(E, gamma, scalar_mult(E, remainder, q ** (e - 1 - j)), q)
            k_i += digit * q ** j
        residues.append(k_i)
        moduli.append(q ** e)

    modulus = math.prod(moduli)
    k = sum(r * (modulus // m) * pow(modulus // m, -1, m) for r, m in zip(residues, moduli)) % modulus
    return k, modulus

In [9]:
E = Curve(a=2, b=3, p=310717010502520989590157367261876774703)
G = (179210853392303317793440285562762725654, 105268671499942631758568591033409611165)
A = (280810182131414898730378982766101210916, 291506490768054478159835604632710368904)  # Alice's public key
B = (272640099140026426377756188075937988094, 51062462309521034358726608268084433317)   # Bob's public key
data = {'iv': '07e2628b590095a5e332d397b8a59aa7', 'encrypted_flag': '8220b7c47b36777a737f5ef9caa2814cf20c1c1ef496ec21a9b4833da24a008d0870d3ac3a6ad80065c138a2ed6136af'}

N = curve_order(E)
print("Curve order factorization:", factor(N))

n = point_order(E, G, N)
n_A, _ = pohlig_hellman(E, G, A, n)
assert scalar_mult(E, G, n_A) == A

print(decrypt_flag(scalar_mult(E, B, n_A)[0], data["iv"], data["encrypted_flag"]))

Curve order factorization: [(2, 2), (3, 7), (139, 1), (165229, 1), (31850531, 1), (270778799, 1), (179317983307, 1)]


crypto{n07_4ll_curv3s_4r3_s4f3_curv3s}


## Exceptional Curves

This 512-bit curve has prime order, which defeats Pohlig-Hellman. But its order is **equal to `p`**:
the curve is *anomalous*, and **Smart's attack** solves its ECDLP in polynomial time.

The idea is to lift the problem to the `p`-adic numbers, where `p·P` is no longer the point at infinity but a point
very close to it. Points near infinity are mapped to `pℤ_p` by the `p`-adic elliptic logarithm `ψ(P) = -x/y`,
which is a group homomorphism. Working modulo `p²` is enough:

1. Hensel-lift `G` and `Q = k·G` to points `G'`, `Q'` of the curve over `ℤ/p²ℤ`.
2. Compute `p·G'` and `p·Q'`. They reduce to `O` modulo `p`, so their `ψ` values are multiples of `p`.
3. Since `ψ(p·Q') = k·ψ(p·G')`, we get `k = (ψ(p·Q') / p) / (ψ(p·G') / p) mod p`.

Projective coordinates `(X : Y : Z)` (with `x = X/Z`, `y = Y/Z`) avoid divisions by multiples of `p` during step 2,
and give `ψ = -X/Y` directly.

In [10]:
def smart_attack(E, G, Q):
    """Solve Q = k*G on an anomalous curve (#E = p)."""
    p, a, b = E.p, E.a, E.b
    p2 = p * p

    def hensel_lift(P):
        # find y' = y + t*p such that y'^2 = x^3 + ax + b (mod p^2)
        x, y = P
        f = y * y - (x ** 3 + a * x + b)
        t = (-f // p) * pow(2 * y, -1, p) % p
        return (x, y + t * p, 1)

    def add(P, Q):
        # projective addition and doubling over Z/p^2
        X1, Y1, Z1 = P
        X2, Y2, Z2 = Q
        if Z1 % p2 == 0:
            return Q
        if Z2 % p2 == 0:
            return P
        U1, U2 = Y2 * Z1 % p2, Y1 * Z2 % p2
        V1, V2 = X2 * Z1 % p2, X1 * Z2 % p2
        if V1 == V2:
            if U1 != U2:
                return (0, 1, 0)
            W = (a * Z1 * Z1 + 3 * X1 * X1) % p2
            S = Y1 * Z1 % p2
            B_ = X1 * Y1 * S % p2
            H = (W * W - 8 * B_) % p2
            return (2 * H * S % p2, (W * (4 * B_ - H) - 8 * Y1 * Y1 * S * S) % p2, 8 * S ** 3 % p2)
        U, V, W = (U1 - U2) % p2, (V1 - V2) % p2, Z1 * Z2 % p2
        A_ = (U * U * W - V ** 3 - 2 * V * V * V2) % p2
        return (V * A_ % p2, (U * (V * V * V2 - A_) - V ** 3 * U2) % p2, V ** 3 * W % p2)

    def mult(P, n):
        R = (0, 1, 0)
        while n:
            if n & 1:
                R = add(R, P)
            P = add(P, P)
            n >>= 1
        return R

    def psi_over_p(P):
        X, Y, Z = mult(hensel_lift(P), p)
        assert Z % p == 0  # p*P reduces to the point at infinity
        return (-X * pow(Y, -1, p2)) % p2 // p

    return psi_over_p(Q) * pow(psi_over_p(G), -1, p) % p

In [11]:
E = Curve(
    a=0x5e009506fcc7eff573bc960d88638fe25e76a9b6c7caeea072a27dcd1fa46abb15b7b6210cf90caba982893ee2779669bac06e267013486b22ff3e24abae2d42,
    b=0x2ce7d1ca4493b0977f088f6d30d9241f8048fdea112cc385b793bce953998caae680864a7d3aa437ea3ffd1441ca3fb352b0b710bb3f053e980e503be9a7fece,
    p=0xa15c4fb663a578d8b2496d3151a946119ee42695e18e13e90600192b1d0abdbb6f787f90c8d102ff88e284dd4526f5f6b6c980bf88f1d0490714b67e8a2a2b77,
)
G = (3034712809375537908102988750113382444008758539448972750581525810900634243392172703684905257490982543775233630011707375189041302436945106395617312498769005, 4986645098582616415690074082237817624424333339074969364527548107042876175480894132576399611027847402879885574130125050842710052291870268101817275410204850)
A = (4748198372895404866752111766626421927481971519483471383813044005699388317650395315193922226704604937454742608233124831870493636003725200307683939875286865, 2421873309002279841021791369884483308051497215798017509805302041102468310636822060707350789776065212606890489706597369526562336256272258544226688832663757)
B = (0x7f0489e4efe6905f039476db54f9b6eac654c780342169155344abc5ac90167adc6b8dabacec643cbe420abffe9760cbc3e8a2b508d24779461c19b20e242a38, 0xdd04134e747354e5b9618d8cb3f60e03a74a709d4956641b234daa8a65d43df34e18d00a59c070801178d198e8905ef670118c15b0906d3a00a662d3a2736bf)
data = {'iv': '719700b2470525781cc844db1febd994', 'encrypted_flag': '335470f413c225b705db2e930b9d460d3947b3836059fb890b044e46cbb343f0'}

assert curve_order(E) == E.p  # anomalous curve

n_A = smart_attack(E, G, A)
assert scalar_mult(E, G, n_A) == A

print(decrypt_flag(scalar_mult(E, B, n_A)[0], data["iv"], data["encrypted_flag"]))

crypto{H3ns3l_lift3d_my_fl4g!}


## Micro Transmissions

The private keys are only **64 bits** long, while the order of `G` is 256 bits. Its factorization contains small
primes and two large ones. Running Pohlig-Hellman on the small factors only recovers `n_A mod M`, where `M` is the
product of the corresponding prime powers. As soon as `M > 2⁶⁴`, this residue **is** the private key.

Alice only sends an x-coordinate, so the lifted point may be `-A` instead of `A`: we get `±n_A mod M`, and keep the
candidate below `2⁶⁴`.

In [12]:
E = Curve(a=1, b=4, p=99061670249353652702595159229088680425828208953931838069069584252923270946291)
G = (43190960452218023575787899214023014938926631792651638044680168600989609069200, 20971936269255296908588589778128791635639992476076894152303569022736123671173)
A_x = 87360200456784002948566700858113190957688355783112995047798140117594305287669
B_x = 6082896373499126624029343293750138460137531774473450341235217699497602895121
data = {'iv': 'ceb34a8c174d77136455971f08641cc5', 'encrypted_flag': 'b503bf04df71cfbd3f464aec2083e9b79c825803a4d4a43697889ad29eb75453'}

n = point_order(E, G, curve_order(E))
print("Order of G:", factor(n))

k, M = pohlig_hellman(E, G, lift_x(E, A_x), n, max_factor=10**13)
print(f"Recovered n_A modulo a {M.bit_length()}-bit modulus")

n_A = next(c for c in (k, M - k) if c < 2**64)
assert scalar_mult(E, G, n_A)[0] == A_x

print(decrypt_flag(scalar_mult(E, lift_x(E, B_x), n_A)[0], data["iv"], data["encrypted_flag"]))

Order of G: [(7, 1), (11, 1), (17, 1), (191, 1), (317, 1), (331, 1), (5221385621, 1), (5397618469, 1), (210071842937040101, 1), (637807437018177170959577732683, 1)]


Recovered n_A modulo a 100-bit modulus
crypto{d0nt_l3t_n_b3_t00_sm4ll}


## Elliptic Nodes

The curve parameters `a` and `b` are secret, and the private key is the flag itself: `Q = flag·G`.

**Recovering the curve.** Both `G` and `Q` satisfy `y² - x³ = ax + b`, a linear system in `a` and `b`.

**The curve is singular.** Its discriminant `4a³ + 27b²` is zero: the cubic has a double root `α` and a simple
root `β`, so the "curve" is `y² = (x - α)²(x - β)`, which has a node at `(α, 0)`. Such a curve is not an elliptic
curve, and its group of non-singular points is isomorphic to the multiplicative group `F_p*` through

$$\phi(x, y) = \frac{y + s(x - \alpha)}{y - s(x - \alpha)}, \qquad s^2 = \alpha - \beta$$

The ECDLP becomes a classic discrete logarithm in `F_p*`, and `p - 1` is smooth enough for PARI's `znlog`
(Pohlig-Hellman with Pollard rho) to solve it.

In [13]:
p = 4368590184733545720227961182704359358435747188309319510520316493183539079703
G = (8742397231329873984594235438374590234800923467289367269837473862487362482, 225987949353410341392975247044711665782695329311463646299187580326445253608)
Q = (2582928974243465355371953056699793745022552378548418288211138499777818633265, 2421683573446497972507172385881793260176370025964652384676141384239699096612)

# 1. Recover a and b
(x1, y1), (x2, y2) = G, Q
a = ((y1 ** 2 - x1 ** 3) - (y2 ** 2 - x2 ** 3)) * pow(x1 - x2, -1, p) % p
b = (y1 ** 2 - x1 ** 3 - a * x1) % p
assert (4 * a ** 3 + 27 * b ** 2) % p == 0  # singular curve

# 2. Find the double root alpha (also a root of the derivative 3x^2 + a) and the simple root beta
roots = [int(r.lift()) for r in pari(f"polrootsmod(x^3 + {a}*x + {b}, {p})")]
alpha = next(r for r in roots if (3 * r * r + a) % p == 0)
beta = -2 * alpha % p  # the three roots alpha, alpha, beta sum to 0
s = sqrt_mod(alpha - beta, p)
assert s is not None  # node with tangent slopes in F_p: the group is isomorphic to F_p*


# 3. Map the points to F_p* and solve the discrete logarithm there
def phi(P):
    x, y = P
    return (y + s * (x - alpha)) * pow(y - s * (x - alpha), -1, p) % p


print("p - 1 =", factor(p - 1))
flag = int(pari(f"znlog(Mod({phi(Q)}, {p}), Mod({phi(G)}, {p}))"))
print(long_to_bytes(flag).decode())

p - 1 = [(2, 1), (2561822865103, 1), (3310993114631, 1), (3782003969119, 1), (3877604210317, 1), (4112912710789, 1), (4269421018381, 1)]
crypto{s1ngul4r_s1mplif1c4t1on}


## Moving Problems

The **MOV attack** (Menezes-Okamoto-Vanstone) uses a pairing to move the ECDLP into a finite field.
The Weil pairing `e_m` is bilinear: for a point `R` of order `m`, `e_m(k·G, R) = e_m(G, R)^k`.
So if `Q = k·G`, then `k` is the discrete logarithm of `e_m(Q, R)` in base `e_m(G, R)`.

The pairing takes its values in `F_{p^k}`, where the *embedding degree* `k` is the smallest integer such that the
order of `G` divides `p^k - 1`. For secure curves `k` is astronomically large, but here `k = 2`: the problem moves to
`F_{p²}`, where discrete logarithms are much easier.

Technical detail: the Weil pairing requires the full `m`-torsion to be defined over `F_{p²}`. This is the case for
`m = n / 7` (where `n` is the order of `G`), so we compute `n_A mod m` with the pairing and brute-force the
remaining factor `7`. The discrete logarithm in `F_{p²}` takes a couple of minutes.

In [14]:
E = Curve(a=-35, b=98, p=1331169830894825846283645180581)
G = (479691812266187139164535778017, 568535594075310466177352868412)
A = (1110072782478160369250829345256, 800079550745409318906383650948)
B = (1290982289093010194550717223760, 762857612860564354370535420319)
data = {'iv': 'eac58c26203c04f68d63dc2c58d79aca', 'encrypted_flag': 'bb9ecbd3662d0671fd222ccb07e27b5500f304e3621a6f8e9c815bc8e4e6ee6ebc718ce9ca115cb4e41acb90dbcabb0d'}

n = point_order(E, G, curve_order(E))
k = next(k for k in range(1, 7) if (E.p ** k - 1) % n == 0)
print("Embedding degree:", k)

# Define the curve over F_{p^k} in PARI
pari(f"Ek = ellinit([{E.a}, {E.b}], ffgen({E.p}^{k}, 'w))")
exponent, second_invariant = (int(d) for d in pari("ellgroup(Ek)"))  # E(F_{p^k}) = Z/d1 x Z/d2
m = math.gcd(n, second_invariant)  # full m-torsion is defined over F_{p^k}
print(f"n / m = {n // m}")

# Move G and A into the subgroup of order m, and pick a random point R of order m
pari(f"m = {m}; Gm = ellmul(Ek, [{G[0]}, {G[1]}], {n // m}); Am = ellmul(Ek, [{A[0]}, {A[1]}], {n // m})")
while True:
    pari(f"R = ellmul(Ek, random(Ek), {exponent // m}); eG = ellweilpairing(Ek, Gm, R, m)")
    if int(pari("fforder(eG)")) == m:  # the pairing must not be degenerate
        break

# Solve the discrete logarithm in F_{p^k}, then brute-force the factor n / m
r = int(pari("fflog(ellweilpairing(Ek, Am, R, m), eG, m)"))
n_A = next(r + i * m for i in range(n // m) if scalar_mult(E, G, r + i * m) == A)

print(decrypt_flag(scalar_mult(E, B, n_A)[0], data["iv"], data["encrypted_flag"]))

Embedding degree: 2
n / m = 7
crypto{MOV_attack_on_non_supersingular_curves}


# Parameter Choice 2

## Real Curve Crypto

This time the curve `y² = x³ - x` is defined over the **real numbers** (with 200 digits of precision), and the
private key `N` is a 128-bit AES key such that `P = N·G`.

Over `ℝ`, the connected component of the identity of `E(ℝ)` is isomorphic to a circle `ℝ / ωℤ`, where `ω` is the
real period of the curve. The isomorphism is the *elliptic logarithm* `z`, which turns point multiplication into
ordinary multiplication:

$$z(P) \equiv N \cdot z(G) \pmod \omega \quad\Longleftrightarrow\quad N \cdot z(G) - z(P) - j \cdot \omega = 0 \text{ for some integer } j$$

This is an integer relation between three known real numbers, with small coefficients `(N, -1, -j)`.
Lattice reduction (LLL) finds it: in the lattice spanned by the rows below, `(N, -1, -j, ≈0)` is a very short
vector. `C` is a large scaling constant.

$$\begin{pmatrix} 1 & 0 & 0 & \lfloor C\,z(G) \rceil \\ 0 & 1 & 0 & \lfloor C\,z(P) \rceil \\ 0 & 0 & 1 & \lfloor C\,\omega \rceil \end{pmatrix}$$

In [15]:
data = {"gx": "1.15939524880832589559531697886995971202850341796875", "gy": "0.63171256444643032392780992695525219049591162225190317013257297329770648576821328127894482837966831464341245854394869021441094534520252153121771214144624262228503025100001998247467928012492847829951495", "px": "1052.1869486109503324827555468817188804055933729601321435932864694301534931492427433020783168479195188024409373571681097603398390379320742186401833284576176214641603772370675124838606986281131453644941", "py": "34130.226434169760878074808301335090475271836563983186858174748793301185920507814323732035620490086239117019147179225251544229354849757884152095399018744815803126126630119173317092410331587882005521149", "ciphertext": "a104b68d30a207eabf293324fbde64f8d628fb07068058c1e76e670e7e805fc567f739185bbe6cbb44f09013173ee653", "iv": "485f9a1e4a3b19348367280df13f9e77"}

pari("default(realprecision, 250)")
pari(f'gx = {data["gx"]}; gy = {data["gy"]}; px = {data["px"]}; py = {data["py"]}')

# Elliptic logarithms of G and P, and the real period of the curve
pari("E = ellinit([-1, 0]); zG = real(ellpointtoz(E, [gx, gy])); zP = real(ellpointtoz(E, [px, py])); w = E.omega[1]")

# LLL on the integer relation lattice (PARI's qflll works on columns, hence the transpositions)
reduced = pari("C = 10^190; M = [1, 0, 0, round(C*zG); 0, 1, 0, round(C*zP); 0, 0, 1, round(C*w)]; (M~ * qflll(M~))~")
rows = [[int(reduced[i, j]) for j in range(4)] for i in range(3)]

N = next(abs(row[0]) for row in rows if abs(row[1]) == 1)
print(f"Recovered {N.bit_length()}-bit key")

key = long_to_bytes(N, 16)
cipher = AES.new(key, AES.MODE_CBC, bytes.fromhex(data["iv"]))
print(unpad(cipher.decrypt(bytes.fromhex(data["ciphertext"])), AES.block_size).decode())

Recovered 127-bit key
crypto{real_fields_arent_finite}


# Signatures

## Digestive

The server signs `{"admin": false, "username": "<name>"}` with ECDSA over NIST P-192, but its "hash function" returns
the message unchanged. ECDSA truncates the digest to the bit length of the group order (192 bits), so only the
**first 24 bytes** of the message are signed: `{"admin": false, "userna`.

Everything after that can be changed freely. We append a second `"admin": true` key: Python's JSON parser keeps the
last occurrence of a duplicate key, so the forged message is accepted as an admin request with a valid signature.

In [16]:
from urllib.parse import quote

import requests

BASE_URL = "https://web.cryptohack.org/digestive"

signed = requests.get(f"{BASE_URL}/sign/alice/").json()
forged = signed["msg"][:-1] + ', "admin": true}'
assert forged[:24] == signed["msg"][:24]  # the only part covered by the signature
print(forged)

response = requests.get(f"{BASE_URL}/verify/{quote(forged)}/{signed['signature']}/").json()
print(response["flag"])

{"admin": false, "username": "alice", "admin": true}
crypto{thanx_for_ctf_inspiration_https://mastodon.social/@filippo/109360453402691894}


## Curveball

A reproduction of **CurveBall** (CVE-2020-0601), a 2020 vulnerability in Windows' certificate validation.
The server trusts a certificate when the public key it derives, `Q = d·g`, matches a trusted public key.
But it lets the client supply the **generator** `g` along with `d`, and only rejects `d = ±1`.

We pick `d = 2` and `g = 2⁻¹·Q_bing`, where `2⁻¹` is the inverse of 2 modulo the group order.
Then `d·g = Q_bing`: the server believes we hold bing.com's private key. The fix is to always check that the
generator is the curve's standard one.

In [17]:
import json

from pwn import context, remote

context.log_level = "error"

# NIST P-256
P256 = Curve(
    a=-3,
    b=0x5ac635d8aa3a93e7b3ebbd55769886bc651d06b0cc53b0f63bce3c3e27d2604b,
    p=0xffffffff00000001000000000000000000000000ffffffffffffffffffffffff,
)
G = (0x6b17d1f2e12c4247f8bce6e563a440f277037d812deb33a0f4a13945d898c296, 0x4fe342e2fe1a7f9b8ee7eb4a7c0f9e162bce33576b315ececbb6406837bf51f5)
q = 0xffffffff00000000ffffffffffffffffbce6faada7179e84f3b9cac2fc632551  # order of G

Q_bing = (0x3B827FF5E8EA151E6E51F8D0ABF08D90F571914A595891F9998A5BD49DFA3531, 0xAB61705C502CA0F7AA127DEC096B2BBDC9BD3B4281808B3740C320810888592A)

d = 2
g = scalar_mult(P256, Q_bing, pow(d, -1, q))
assert scalar_mult(P256, g, d) == Q_bing

with remote("socket.cryptohack.org", 13382) as conn:
    conn.recvline()  # welcome banner
    conn.sendline(json.dumps({"private_key": d, "host": "www.bing.com", "curve": "secp256r1", "generator": list(g)}).encode())
    print(json.loads(conn.recvline()))

Hey bing! Tell me about crypto{Curveballing_Microsoft_CVE-2020-0601}


## ProSign 3

A timestamp signing server uses ECDSA on NIST P-192. Its `sign_time` method reads the current time into variables
`m, n`, which **overwrites `n`, the group order**, with the current seconds. The nonce is then drawn with
`randrange(1, n)`, so it is always **smaller than 60**.

From a single signature `(r, s)` of a known message with hash `h`, we try the 59 possible nonces until
`x(k·G) mod q = r`. The ECDSA equation `s·k ≡ h + r·d (mod q)` then gives the private key:

$$d = (s \cdot k - h) \cdot r^{-1} \bmod q$$

and we sign `unlock` ourselves. At second 0 the server crashes (`randrange(1, 0)`), so we simply ask again.

In [18]:
import secrets

# NIST P-192
P192 = Curve(a=-3, b=0x64210519e59c80e70fa7e9ab72243049feb8deecc146b9b1, p=2**192 - 2**64 - 1)
G192 = (0x188da80eb03090f67cbf20eb43a18800f4ff0afd82ff1012, 0x07192b95ffc8da78631011ed6b24cdd573f977a11e794811)
q192 = 0xffffffffffffffffffffffff99def836146bc9b1b4d22831  # order of G192


def sha1_int(msg):
    return bytes_to_long(hashlib.sha1(msg.encode()).digest())


def ecdsa_sign(msg, d):
    k = secrets.randbelow(q192 - 1) + 1
    r = scalar_mult(P192, G192, k)[0] % q192
    s = pow(k, -1, q192) * (sha1_int(msg) + r * d) % q192
    return r, s


with remote("socket.cryptohack.org", 13381) as conn:
    conn.recvline()  # welcome banner
    while True:
        conn.sendline(json.dumps({"option": "sign_time"}).encode())
        signature = json.loads(conn.recvline())
        if "r" in signature:  # no signature at second 0
            break
    print(signature)

    r, s, h = int(signature["r"], 16), int(signature["s"], 16), sha1_int(signature["msg"])
    k = next(k for k in range(1, 60) if scalar_mult(P192, G192, k)[0] % q192 == r)
    d = (s * k - h) * pow(r, -1, q192) % q192
    print("Nonce:", k)

    r_unlock, s_unlock = ecdsa_sign("unlock", d)
    conn.sendline(json.dumps({"option": "verify", "msg": "unlock", "r": hex(r_unlock), "s": hex(s_unlock)}).encode())
    print(json.loads(conn.recvline())["flag"])

{'msg': 'Current time is 10:30', 'r': '0x818a4d308b1cabb74e9e8f2ba8d27c9e1d9d375ab980388f', 's': '0xd49034570a14ee865db2e7d10a67e3d5ed7431b7050cc7df'}
Nonce: 9
crypto{ECDSA_700_345y_70_5cr3wup}


## No Random, No Bias

The flag is hidden as a point `T = d·Q`, where `x(Q)` is the flag and `d` is the ECDSA private key on NIST P-256.
We are given three signatures. The nonces are derived deterministically with SHA-1, so they are only
**160 bits** long instead of 256: they are *biased*.

An ECDSA signature `(r, s)` of a message with hash `h` satisfies `s·k ≡ h + r·d (mod q)`, so each nonce is

$$k_i \equiv t_i \cdot d + u_i \pmod q, \qquad t_i = s_i^{-1} r_i, \quad u_i = s_i^{-1} h_i$$

Recovering `d` from these relations and the knowledge that every `k_i < B = 2^{160}` is an instance of the
**hidden number problem**, solved with LLL. The vector `(k_1, k_2, k_3, B·d/q, B)` is a short vector of the lattice
spanned by the rows of

$$\begin{pmatrix} q & 0 & 0 & 0 & 0 \\ 0 & q & 0 & 0 & 0 \\ 0 & 0 & q & 0 & 0 \\ t_1 & t_2 & t_3 & B/q & 0 \\ u_1 & u_2 & u_3 & 0 & B \end{pmatrix}$$

(multiplied by `q` to keep integer entries). Once `d` is known, `Q = d⁻¹·T`.

In [19]:
# P256, G and q are defined in the Curveball solution above
hidden_flag = (16807196250009982482930925323199249441776811719221084165690521045921016398804, 72892323560996016030675756815328265928288098939353836408589138718802282948311)
public_key = (48780765048182146279105449292746800142985733726316629478905429239240156048277, 74172919609718191102228451394074168154654001177799772446328904575002795731796)
signatures = [
    {'msg': 'I have hidden the secret flag as a point of an elliptic curve using my private key.', 'r': '0x91f66ac7557233b41b3044ab9daf0ad891a8ffcaf99820c3cd8a44fc709ed3ae', 's': '0x1dd0a378454692eb4ad68c86732404af3e73c6bf23a8ecc5449500fcab05208d'},
    {'msg': 'The discrete logarithm problem is very hard to solve, so it will remain a secret forever.', 'r': '0xe8875e56b79956d446d24f06604b7705905edac466d5469f815547dea7a3171c', 's': '0x582ecf967e0e3acf5e3853dbe65a84ba59c3ec8a43951bcff08c64cb614023f8'},
    {'msg': 'Good luck!', 'r': '0x566ce1db407edae4f32a20defc381f7efb63f712493c3106cf8e85f464351ca6', 's': '0x9e4304a36d2c83ef94e19a60fb98f659fa874bfb999712ceb58382e2ccda26ba'},
]


def lll(rows):
    """LLL-reduce the lattice spanned by the given integer rows (using PARI)."""
    M = pari.matrix(len(rows), len(rows[0]), [x for row in rows for x in row]).mattranspose()
    reduced = M * pari.qflll(M)
    return [[int(reduced[i, j]) for i in range(len(rows[0]))] for j in range(len(rows))]


t, u = [], []
for sig in signatures:
    h = bytes_to_long(hashlib.sha1(sig["msg"].encode()).digest())
    r, s = int(sig["r"], 16), int(sig["s"], 16)
    t.append(pow(s, -1, q) * r % q)
    u.append(pow(s, -1, q) * h % q)

B = 2 ** 160
n = len(signatures)
rows = [[q * q if j == i else 0 for j in range(n)] + [0, 0] for i in range(n)]
rows.append([q * t_i for t_i in t] + [B, 0])
rows.append([q * u_i for u_i in u] + [0, q * B])

for v in lll(rows):
    if abs(v[-1]) == q * B:  # the target vector ends with ±q*B
        k_1 = v[0] // q if v[-1] > 0 else -v[0] // q
        d = (k_1 - u[0]) * pow(t[0], -1, q) % q
        if scalar_mult(P256, G, d) == public_key:
            break
else:
    raise RuntimeError("private key not found")

Q = scalar_mult(P256, hidden_flag, pow(d, -1, q))
print(long_to_bytes(Q[0]).decode())

crypto{3mbrac3_r4nd0mn3ss}


# Edwards Curves

## Edwards Goes Degenerate

Points are compressed as `(parity of x, y)` and decompressed by solving the curve equation for `x`.
The decompression has a bug: it checks `x**2 == xsqr` **without reducing modulo `p`**, so the test almost always
fails and `x` is set to `0`.

With `x₁ = x₂ = 0`, the twisted Edwards addition formulas collapse to `(0, y₁·y₂)`.
Scalar multiplication becomes plain exponentiation: `n·(0, y) = (0, yⁿ mod p)`.
The ECDLP turns into a discrete logarithm in `F_p*`, and `p - 1` is smooth.

In [20]:
p = 110791754886372871786646216601736686131457908663834453133932404548926481065303
y0 = 11  # y-coordinate of the base point
y_alice = 109790246752332785586117900442206937983841168568097606235725839233151034058387
y_bob = 45290526009220141417047094490842138744068991614521518736097631206718264930032
data = {'iv': '31068e75b880bece9686243fa4dc67d0', 'encrypted_flag': 'e2ef82f2cde7d44e9f9810b34acc885891dad8118c1d9a07801639be0629b186dc8a192529703b2c947c20c4fe5ff2c8'}

print("p - 1 =", factor(p - 1))
n_A = int(pari(f"znlog(Mod({y_alice}, {p}), Mod({y0}, {p}))"))
assert pow(y0, n_A, p) == y_alice

shared_secret = pow(y_bob, n_A, p)  # y-coordinate of n_A * P_bob
print(decrypt_flag(shared_secret, data["iv"], data["encrypted_flag"]))

p - 1 = [(2, 1), (33682721, 1), (33964681, 1), (42126503, 1), (46340641, 1), (49189373, 1), (49309751, 1), (50173309, 1), (56252591, 1), (58172209, 1), (62285989, 1)]


crypto{degenerates_will_never_keep_a_secret}


# Side Channels

## Montgomery's Ladder

Double-and-add leaks the bits of the secret scalar through timing, since it only adds when a bit is set.
Montgomery's ladder performs exactly one addition and one doubling per bit, keeping the invariant `R1 - R0 = G`:

- bit `0`: `(R0, R1) ← (2·R0, R0 + R1)`
- bit `1`: `(R0, R1) ← (R0 + R1, 2·R1)`

The challenge uses Curve25519, a Montgomery curve `By² = x³ + Ax² + x` with `A = 486662`, `B = 1`,
`p = 2²⁵⁵ - 19`, and base point `x(G) = 9`. On Montgomery curves the ladder can work on x-coordinates only:
because `R1 - R0 = G` is fixed, `x(R0 + R1)` can be computed from `x(R0)`, `x(R1)` and `x(G)`:

$$x(2P) = \frac{(x^2 - 1)^2}{4x(x^2 + Ax + 1)}, \qquad x(P + Q) = \frac{(x_P x_Q - 1)^2}{x_{P-Q}(x_P - x_Q)^2}$$

In [21]:
A = 486662
p25519 = 2**255 - 19


def x_double(x):
    num = (x * x - 1) ** 2
    den = 4 * x * (x * x + A * x + 1)
    return num * pow(den, -1, p25519) % p25519


def x_add(xP, xQ, x_diff):
    num = (xP * xQ - 1) ** 2
    den = x_diff * (xP - xQ) ** 2
    return num * pow(den, -1, p25519) % p25519


def montgomery_ladder(xG, k):
    """Compute x(k·G) using x-coordinate-only arithmetic."""
    R0, R1 = xG, x_double(xG)
    for i in reversed(range(k.bit_length() - 1)):  # skip the leading 1 bit
        if (k >> i) & 1:
            R0, R1 = x_add(R0, R1, xG), x_double(R1)
        else:
            R0, R1 = x_double(R0), x_add(R0, R1, xG)
    return R0


print(montgomery_ladder(9, 0x1337c0decafe))

49231350462786016064336756977412654793383964726771892982507420921563002378152


## Double and Broken

This is exactly the weakness the ladder avoids. The private key (the flag) is used in a double-and-add scalar
multiplication on secp256k1, and we are given 50 noisy **power traces**: one measurement per bit of the key.
Iterations that perform a doubling *and* an addition consume more power than doubling-only iterations.

Averaging the 50 traces removes most of the noise and reveals two clear levels. Each value above the midpoint is a
`1` bit. The right-to-left double-and-add processes the key starting from the least significant bit, so the bit
string is reversed before being converted back to bytes.

In [22]:
import numpy as np

with open("files/double_and_broken_traces.json") as f:
    traces = np.array(json.load(f))
print("Traces:", traces.shape)

mean = traces.mean(axis=0)
threshold = (mean.min() + mean.max()) / 2
print("Average power of the first iterations:", mean[:8].round())

bits = "".join("1" if level > threshold else "0" for level in mean)
d = int(bits[::-1], 2)  # least significant bit first
print(long_to_bytes(d).decode())

Traces: (50, 359)
Average power of the first iterations: [155.  96. 156. 151. 148. 156. 154. 101.]
crypto{Sid3_ch4nn3ls_c4n_br34k_s3cur3_curv3s}
